In [65]:
import pandas as pd
import numpy as np

# Giả sử 'df_raw' là dataframe chứa 31 thuộc tính ban đầu của bạn
# Load dataset

DATA_PATH = "../../data/processed/phone_specs_preprocessed_v2.csv"

df = pd.read_csv(DATA_PATH)
display(df.isna().sum())

product_name                0
brand                       0
url                         0
price_vnd                   0
price_segment               0
promotion                  19
phone_type                  0
os_family                   0
os_version                 74
chip_name                   0
chip_brand                  0
screen_size_inch            1
panel                       0
panel_group                 0
refresh_rate_hz             0
resolution_width            0
resolution_height           0
resolution_total_pixels     0
ram_gb                      0
storage_gb                  0
battery_mah                 0
fast_charge_w               0
front_camera_mp             3
rear_main_camera_mp         7
rear_camera_count           7
max_video_resolution        0
fps_at_max_resolution       0
video_resolution_rank       0
weight_g                    0
ip_status                   0
dtype: int64

## Rating màn hình

In [66]:
df['screen_size_fixed'] = df.groupby('price_segment')['screen_size_inch'].transform(lambda x: x.fillna(x.median()))

features = ['screen_size_fixed', 'panel_group', 'resolution_total_pixels']
display(df[features].isna().sum())

screen_size_fixed          0
panel_group                0
resolution_total_pixels    0
dtype: int64

In [67]:
import os
os.environ['OMP_NUM_THREADS'] = '2'  # Fix KMeans MKL memory leak on Windows
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Giả lập tập dữ liệu lớn hơn một chút để K-Means hoạt động ổn định
# data = {
#     'screen_size': [15.6, 14.0, 27.0, 13.3, 15.6, 11.6, 32.0, 14.0],
#     'panel_group': ['TN', 'IPS', 'IPS', 'OLED', 'VA', 'TN', 'Mini-LED', 'OLED'],
#     'resolution_total_pixels': [2073600, 2073600, 8294400, 5184000, 2073600, 1049088, 8294400, 3686400]
# }
# df = pd.DataFrame(data)

# 2. Xử lý thuộc tính chữ (Ordinal Encoding dựa trên thứ tự chất lượng)
panel_map = {'Unknown':0, 'LCD': 2, 'AMOLED': 3}
df['panel_score'] = df['panel_group'].map(panel_map)

#Chọn các tính năng để đưa vào K-Means
features = ['screen_size_fixed', 'panel_score', 'resolution_total_pixels']
X = df[features].fillna(df[features].median())  # Defensive: tránh NaN vào StandardScaler

# 3. CHUẨN HÓA DỮ LIỆU (Bắt buộc phải có với K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. CHẠY K-MEANS (Chia làm 4 cụm tương ứng 4 Tiers)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_raw'] = kmeans.fit_predict(X_scaled)

# 5. ĐỊNH DANH LẠI THỨ TỰ (Sắp xếp cụm từ thấp đến cao theo độ phân giải trung bình)
# Tính độ phân giải trung bình của từng cụm thô
cluster_order = df.groupby('cluster_raw')['resolution_total_pixels'].mean().sort_values().index

# Tạo bản đồ ánh xạ từ cụm ngẫu nhiên sang tên Tier chuẩn
tier_names = [1,2,3,4]
cluster_mapping = {raw_cluster: tier_names[i] for i, raw_cluster in enumerate(cluster_order)}

# Gán nhãn cuối cùng
df['display_tier'] = df['cluster_raw'].map(cluster_mapping)

# Xem kết quả kiểm chứng
print(df[df['price_segment']=='flagship'][['screen_size_inch', 'panel_group', 'resolution_height', 'display_tier']])

     screen_size_inch panel_group  resolution_height  display_tier
3                6.20      AMOLED             3200.0             2
9                8.12      AMOLED             2760.0             4
11               6.30      AMOLED             2340.0             2
17               6.90      AMOLED             3200.0             3
19               6.10      AMOLED             2532.0             2
..                ...         ...                ...           ...
374              6.60      AMOLED             2340.0             3
378              8.00      AMOLED             2184.0             4
381              7.60      AMOLED             2176.0             4
384              6.70      AMOLED             2778.0             3
386              7.60      AMOLED             2176.0             4

[111 rows x 4 columns]


## Rating chip

In [68]:
print(df['chip_name'].head())

0           Helio G85 (12nm)
1         Snapdragon 6 Gen 3
2                 Exynos 850
3         Exynos 990 (7 nm+)
4    Qualcomm Snapdragon 685
Name: chip_name, dtype: str


In [69]:
import re
path_chip_rating = '../../data/raw/processor/processors_ratings.csv'
df_chip_rating = pd.read_csv(path_chip_rating)
# ---------------------------------------------------------
# 1. BƯỚC CHUẨN BỊ (Làm sạch cơ bản)
# ---------------------------------------------------------
# Chúng ta vẫn cần một chút làm sạch nhẹ để xóa các ký tự cản trở như ®, ™ và đưa về chữ thường
def light_clean(text):
    if pd.isna(text): 
        return ""
    text = str(text).lower()
    text = text.replace('(', '').replace(')', '')
    text = text.replace('+','')
    
    # 2. Xóa các ký tự thương hiệu như ®, ™
    text = re.sub(r'[®™]', '', text) 

    text = text.replace('thế hệ thứ', 'gen')
    text = text.replace('thế hệ', 'gen')

    
    
    # 3. Dọn dẹp khoảng trắng thừa (biến nhiều khoảng trắng liên tiếp thành 1 khoảng trắng)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def normalize_aliases(text):
    if '8 elite' in text:
        if 'gen 5' in text:
            return 'snapdragon 8 elite gen 5'
        else:
            # Ép tất cả các bản 8 elite còn lại (bản trơn, 3nm, for galaxy, gen 4...) về đúng tên chuẩn
            return 'snapdragon 8 elite gen 4'
    if 'apple a15' in text or text == 'a15': return 'a15 bionic'
    if 'chip a18' in text or text == 'a18': return 'apple a18'
    if 't612' in text: return 'tiger t612'
    if 't616' in text: return 'tiger t616'
    if 't606' in text: return 'tiger t606'
    if 'g99 ultimate' in text: return 'helio g99 ultimate'
    if 'g99' in text and 'helio' not in text: return 'helio g99'
    if 'g100' in text and 'helio' not in text: return 'helio g100'
    if '7025' in text and 'ultra' in text: return 'dimensity 7025 ultra'
    if '8895' in text: return 'exynos 8895'
    return text

# Áp dụng cho DataFrame chính
df['chip_clean'] = df['chip_name'].apply(light_clean).apply(normalize_aliases)

# Chuẩn hóa cột file CSV
df_chip_rating.columns = df_chip_rating.columns.str.lower()
df_chip_rating['processor_clean'] = df_chip_rating['processor'].apply(light_clean)
df_chip_rating['brand_clean'] = df_chip_rating['brand'].astype(str).str.lower().str.strip()
# ---------------------------------------------------------
# 2. TẠO TỪ ĐIỂN VÀ TÍNH ĐIỂM MIN TỰ ĐỘNG TỪ CỘT BRAND
# ---------------------------------------------------------
# Từ điển Chip
chip_df_grouped = df_chip_rating.groupby('processor_clean')['rating'].max().reset_index()
chip_dict = dict(zip(chip_df_grouped['processor_clean'], chip_df_grouped['rating']))
sorted_chip_names = sorted(chip_dict.keys(), key=len, reverse=True)

# Tự động tạo Dictionary lấy điểm Min của cả 8 hãng có trong file CSV
# Kết quả sẽ có dạng: {'qualcomm': 24, 'apple': 24, 'mediatek': 23, 'samsung': 23, ...}
brand_mins = df_chip_rating.groupby('brand_clean')['rating'].min().to_dict()

# ---------------------------------------------------------
# 3. QUÉT CHUỖI VÀ GÁN ĐIỂM (CẬP NHẬT 8 HÃNG)
# ---------------------------------------------------------
def get_final_chip_score(messy_name):
    if not messy_name or messy_name == 'unknown':
        return 0
        
    # 3.1: Dò khớp tên chip chính xác
    for clean_name in sorted_chip_names:
        if clean_name in messy_name:
            return chip_dict[clean_name]
            
    # 3.2: Dò từ khóa (Keywords) để nhận diện 8 hãng và giáng cấp xuống Min score
    if any(k in messy_name for k in ['qualcomm', 'snapdragon', 'sdm']): 
        return brand_mins.get('qualcomm', 0)
        
    if any(k in messy_name for k in ['apple', 'bionic', 'chip a']): 
        return brand_mins.get('apple', 0)
        
    if any(k in messy_name for k in ['mediatek', 'dimensity', 'helio', 'mt6', 'mt8']): 
        return brand_mins.get('mediatek', 0)
        
    if any(k in messy_name for k in ['samsung', 'exynos']): 
        return brand_mins.get('samsung', 0)
        
    if any(k in messy_name for k in ['xiaomi', 'surge', 'xring']): 
        return brand_mins.get('xiaomi', 0)
        
    if any(k in messy_name for k in ['hisilicon', 'kirin']): 
        return brand_mins.get('hisilicon', 0)
        
    if any(k in messy_name for k in ['google', 'tensor']): 
        return brand_mins.get('google', 0)
        
    if any(k in messy_name for k in ['unisoc', 'tiger', 'spreadtrum', 'ums', ' t6', ' t7']): 
        return brand_mins.get('unisoc', 0)
    
    # 3.3: Dữ liệu rác hoàn toàn (Ví dụ: "8 nhân", "Mali-G57")
    return 0

# ---------------------------------------------------------
# 4. THỰC THI VÀ DỌN DẸP
# ---------------------------------------------------------
df['chip_rating'] = df['chip_clean'].apply(get_final_chip_score)
df = df.drop(columns=['chip_clean'])

# Kiểm tra dữ liệu rớt đài
zero_score_chips = df[df['chip_rating'] == 0]['chip_name'].unique()
print("Dữ liệu rác (0 điểm):", zero_score_chips)

Dữ liệu rác (0 điểm): <ArrowStringArray>
[                                    '8 nhân',
 '10 nhân, 3.3 GHz, 2.74GHz, 2.36GHz, 1.8GHz',
                                      'T7200',
                                   'Mali-G57',
                                    'Unknown']
Length: 5, dtype: str


# Camera scoring

In [70]:
def print_missing_percent(df):
    missing_count = df.isnull().sum()

    # Tính tỷ lệ phần trăm giá trị thiếu ở từng cột
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    # Gộp thành một DataFrame thống kê cho trực quan
    missing_summary = pd.DataFrame({
        'Số lượng thiếu': missing_count,
        'Tỷ lệ phần trăm (%)': missing_percentage
    })
    print(missing_summary)
rating_path = "../../data/raw/rating/phone_ratings.csv" 
rating_df = pd.read_csv(rating_path)
print("--- THỐNG KÊ GIÁ TRỊ THIẾU ---")
print_missing_percent(rating_df)
print("\n" + "="*50 + "\n")


# =====================================================================
# BƯỚC 2: DROP CÁC HÀNG KHÔNG CÓ CẢ CAMERA_RATING VÀ DISPLAY_RATING
# =====================================================================

print("--- XỬ LÝ DROP DỮ LIỆU THIẾU ---")
print(f"Số lượng dòng ban đầu: {len(rating_df)}")

# subset: chỉ định các cột cần kiểm tra
# how='all': xóa nếu CẢ 2 cột đều không có giá trị (NaN)
# inplace=False (mặc định) sẽ trả về một DataFrame mới, bạn có thể gán lại vào rating_df nếu muốn ghi đè
rating_df_cleaned = rating_df.dropna(subset=['camera_rating', 'display_rating'], how='all')

print(f"Số lượng dòng sau khi drop: {len(rating_df_cleaned)}")

# Hiển thị dữ liệu sau khi đã làm sạch
print("\nDữ liệu sau khi làm sạch:")
missing_p = (rating_df_cleaned.isnull().sum() / len(rating_df_cleaned)) * 100
print_missing_percent(rating_df_cleaned)

--- THỐNG KÊ GIÁ TRỊ THIẾU ---
                Số lượng thiếu  Tỷ lệ phần trăm (%)
product_name                 0             0.000000
camera_rating              154            37.378641
display_rating             237            57.524272


--- XỬ LÝ DROP DỮ LIỆU THIẾU ---
Số lượng dòng ban đầu: 412
Số lượng dòng sau khi drop: 286

Dữ liệu sau khi làm sạch:
                Số lượng thiếu  Tỷ lệ phần trăm (%)
product_name                 0             0.000000
camera_rating               28             9.790210
display_rating             111            38.811189


In [71]:
import re
from rapidfuzz import process, fuzz

# 1. Tiền xử lý tên máy: strip GB/TB/4G/5G/VN/A
def clean_phone_name(name):
    if pd.isna(name): return ""
    name = re.sub(r'\bApple\s+iPhone\b', 'iPhone', str(name), flags=re.I)
    name = re.sub(r'\b\d+GB\b', '', name, flags=re.I)
    name = re.sub(r'\b\d+TB\b', '', name, flags=re.I)
    name = re.sub(r'\b(4G|5G|LTE)\b', '', name, flags=re.I)
    name = re.sub(r'\b(VN/?A|Chính hãng|I Chính hãng)\b', '', name, flags=re.I)
    name = re.sub(r'\s+', ' ', name).strip()
    return name.lower()

# 2. Validate model number: sau khi strip GB/TB, số còn lại = model id
# Nếu 2 tên đều có số nhưng không chung số nào → sai model → reject
# Ví dụ: S26 vs S23 → {26} ∩ {23} = ∅ → reject
#        X9 vs X6   → {9}  ∩ {6}  = ∅ → reject
#        Note13 vs Note13 → {13} ∩ {13} ≠ ∅ → accept
def validate_model_match(orig_clean, matched_clean):
    orig_nums    = set(re.findall(r'\d+', orig_clean))
    matched_nums = set(re.findall(r'\d+', matched_clean))
    if not orig_nums or not matched_nums:
        return True  # tên không có số (iPhone SE...) → chấp nhận
    return bool(orig_nums & matched_nums)

# 3. Áp dụng fuzzy match + validate
rating_path = '../../data/raw/rating/phone_ratings.csv'
rating_df = pd.read_csv(rating_path)
rating_df['clean_name'] = rating_df['product_name'].apply(clean_phone_name)
rating_choices = rating_df['clean_name'].tolist()
rating_scores  = rating_df['camera_rating'].tolist()

df['clean_name'] = df['product_name'].apply(clean_phone_name)

def get_camera_rating(clean_name):
    if not clean_name: return np.nan
    result = process.extractOne(clean_name, rating_choices, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= 75 and validate_model_match(clean_name, rating_choices[result[2]]):
        return rating_scores[result[2]]
    return np.nan

df_final = df.copy()
df_final['camera_rating_match'] = df_final['clean_name'].apply(get_camera_rating)
print(f"Direct match hợp lệ : {df_final['camera_rating_match'].notna().sum()}")
print(f"Fallback (brand+seg) : {df_final['camera_rating_match'].isna().sum()}")


Direct match hợp lệ : 163
Fallback (brand+seg) : 224


In [72]:
# Kiểm tra tỷ lệ match trực tiếp vs fallback
direct = df_final['camera_rating_match'].notna().sum()
fallback = df_final['camera_rating_match'].isna().sum()
print(f'Direct match: {direct} ({direct/len(df_final)*100:.1f}%)')
print(f'Fallback    : {fallback} ({fallback/len(df_final)*100:.1f}%)')


Direct match: 163 (42.1%)
Fallback    : 224 (57.9%)


In [73]:
# Fallback 1: brand + segment median (chính xác hơn brand-only)
df_final['camera_rating_final'] = df_final['camera_rating_match'].fillna(
    df_final.groupby(['brand','price_segment'])['camera_rating_match'].transform('median'))

# Fallback 2: brand median
df_final['camera_rating_final'] = df_final['camera_rating_final'].fillna(
    df_final.groupby('brand')['camera_rating_match'].transform('median'))

# Fallback 3: global median
df_final['camera_rating_final'] = df_final['camera_rating_final'].fillna(
    df_final['camera_rating_final'].median())

print(f"camera_rating_final NaN: {df_final['camera_rating_final'].isna().sum()}")


camera_rating_final NaN: 0


## KBRS scoring

In [74]:
# ip_status trong phone_specs_preprocessed_v2.csv đã là integer 0/1
# (0 = không chống nước, 1 = có chứng nhận IP)
# Lỗi cũ: astype(str) + == 'certified' → luôn False → tất cả phones thành 0
# Fix: giữ nguyên giá trị, chỉ ép kiểu int
df_final['ip_status'] = df_final['ip_status'].astype(int)
print(f"ip_status: {df_final['ip_status'].value_counts().to_dict()}")
# Kết quả đúng: {1: 269, 0: 118}

ip_status: {1: 269, 0: 118}


In [75]:
# Điền weight_g bằng trung vị của phân khúc giá thay vì hardcode 194g
# Lý do: flagship median=211g, budget=192g, hardcode 194 điền sai cho flagship

# Xử lý outlier: weight_g < 100g là lỗi scraping (smartphone không thể nhẹ vậy)
df_final.loc[df_final['weight_g'] < 100, 'weight_g'] = np.nan

segment_weight_median = df_final.groupby('price_segment')['weight_g'].transform('median')
df_final['weight_g'] = df_final['weight_g'].fillna(segment_weight_median)

conditions = [
    df_final['weight_g'] < 185.0,
    (df_final['weight_g'] >= 185.0) & (df_final['weight_g'] < 210.0)
]
df_final['weight_tier'] = np.select(conditions, [1, 2], default=3)

print(df_final['weight_tier'].value_counts())
print(f"weight_g range: {df_final['weight_g'].min():.0f}g – {df_final['weight_g'].max():.0f}g")
print(f"OPPO X9 Ultra weight_tier: {df_final[df_final['product_name'].str.contains('Find X9 Ultra', na=False)]['weight_tier'].values}")


weight_tier
2    227
3     85
1     75
Name: count, dtype: int64
weight_g range: 135g – 263g
OPPO X9 Ultra weight_tier: [3]


In [76]:
final_export_features = [
    # ---------------------------------------------------------
    # NHÓM 1: THÔNG TIN ĐỊNH DANH & THƯƠNG MẠI (Hiển thị + Lọc segment)
    # ---------------------------------------------------------
    'product_name',       # Tên sản phẩm
    'brand',              # Thương hiệu
    'url',                # Đường dẫn mua hàng
    'price_vnd',          # Giá tiền bằng số để lọc khoảng giá
    'price_segment',      # Phân khúc giá (budget, mid-range, flagship)
    'promotion',          # Thông tin khuyến mãi
    
    # ---------------------------------------------------------
    # NHÓM 2: BỘ CHỈ SỐ ĐIỂM TOÀN CỤC (Dùng để Rank kết quả)
    # ---------------------------------------------------------
    'score_perf',         # Điểm hiệu năng tổng hợp
    'score_cam',          # Điểm camera tổng hợp
    'score_batt',         # Điểm pin & sạc tổng hợp
    'score_disp',         # Điểm màn hình tổng hợp
    
    # ---------------------------------------------------------
    # NHÓM 3: CÁC BIẾN ĐIỀU KIỆN CỨNG (Dùng cho Hard Filters trong engine)
    # ---------------------------------------------------------
    'weight_tier',           # Phân loại cân nặng (1: Nhẹ<185g, 2: Vừa, 3: Nặng≥210g)
    'ip_status',             # Chống nước (1=có chứng nhận IP, 0=không)
    'display_tier',          # Tier màn hình từ KMeans (1-4, dùng cho Giai_Tri_MXH filter)
    'video_resolution_rank', # Rank độ phân giải video (4=4K+, dùng cho Chup_Hinh_Quay_Phim filter)
    
    # ---------------------------------------------------------
    # NHÓM 4: CHI TIẾT PHẦN CỨNG (Hiển thị bảng thông số cho user)
    # ---------------------------------------------------------
    # Cấu hình chính
    'chip_name', 'ram_gb', 'storage_gb', 'os_family', 'os_version',
    
    # Màn hình
    'screen_size_inch', 'panel_group', 'refresh_rate_hz', 'resolution_width', 'resolution_height',
    
    # Pin & Sạc
    'battery_mah', 'fast_charge_w',
    
    # Camera chi tiết
    'front_camera_mp', 'rear_main_camera_mp', 'rear_camera_count', 'max_video_resolution', 'fps_at_max_resolution',
    
    # Thiết kế gốc
    'weight_g',           # Cân nặng số thực (ví dụ: 187g)
]

Nhóm 1: Điều kiện cứng (Hard Filters - Bắt buộc thỏa mãn)

- Giá tiền tối đa.

- Phân khúc ưu tiên: Giá rẻ / Flagship cao cấp / Bất kỳ.

Nhóm 2: Mục đích sử dụng (Soft Constraints / Scoring - Dùng để xếp hạng)

- Liên lạc và cơ bản.

- Chơi game.

- Giải trí và Mạng xã hội.

- Chụp hình, quay phim.

- Pin trâu, sạc nhanh.

In [77]:
def calculate_feature_scores(df):
    """
    Hàm tổng hợp các biến vật lý thành 4 cột điểm (thang điểm 10) cho KBRS.
    Tất cả missing values đã được xử lý ở các bước trên → không còn NaN.
    """
    df_scored = df.copy()

    def min_max_scale(series):
        return (series - series.min()) / (series.max() - series.min() + 1e-9)

    # ---------------------------------------------------------
    # 1. TÍNH ĐIỂM HIỆU NĂNG (score_perf)
    # ---------------------------------------------------------
    # chip_rating được lấy từ benchmark thực tế (AnTuTu/Geekbench) chạy trực tiếp
    # trên thiết bị → đã bao gồm iOS optimization cho Apple Silicon.
    # Vì vậy RAM đóng vai trò khác nhau giữa 2 hệ điều hành:
    #   Android: RAM ảnh hưởng rõ đến multitasking, app retention → weight cao hơn
    #   Apple  : iOS quản lý bộ nhớ hiệu quả hơn, chip_rating đã capture phần lớn
    #            performance → RAM weight thấp hơn, tránh double-count với chip
    norm_chip    = min_max_scale(df_scored['chip_rating'])
    norm_ram     = min_max_scale(df_scored['ram_gb'])
    norm_storage = min_max_scale(df_scored['storage_gb'])

    is_apple = df_scored['brand'] == 'Apple'
    df_scored['score_perf'] = np.where(
        is_apple,
        (norm_chip * 0.65 + norm_ram * 0.15 + norm_storage * 0.20) * 10,  # Apple
        (norm_chip * 0.50 + norm_ram * 0.30 + norm_storage * 0.20) * 10   # Android
    )

    # ---------------------------------------------------------
    # 2. TÍNH ĐIỂM CAMERA (score_cam)
    # ---------------------------------------------------------
    df_scored['score_cam'] = min_max_scale(df_scored['camera_rating_final']) * 10

    # ---------------------------------------------------------
    # 3. TÍNH ĐIỂM PIN & SẠC (score_batt)
    # battery_mah và fast_charge_w đã được điền ở bước trước
    # ---------------------------------------------------------
    norm_battery = min_max_scale(df_scored['battery_mah'])
    norm_charge  = min_max_scale(df_scored['fast_charge_w'])
    df_scored['score_batt'] = (norm_battery * 0.5 + norm_charge * 0.5) * 10

    # ---------------------------------------------------------
    # 4. TÍNH ĐIỂM MÀN HÌNH (score_disp)
    # ---------------------------------------------------------
    norm_dis = min_max_scale(df_scored['display_tier'])
    norm_hz  = min_max_scale(df_scored['refresh_rate_hz'])
    df_scored['score_disp'] = (norm_dis * 0.6 + norm_hz * 0.4) * 10

    score_cols = ['score_perf', 'score_cam', 'score_batt', 'score_disp']
    df_scored[score_cols] = df_scored[score_cols].round(2)

    print('NaN trong scores:')
    print(df_scored[score_cols].isna().sum())

    return df_scored[final_export_features]


In [78]:
df_ready_for_kbrs = calculate_feature_scores(df_final)

NaN trong scores:
score_perf    0
score_cam     0
score_batt    0
score_disp    0
dtype: int64


In [79]:
# =====================================================================
# SENSITIVITY ANALYSIS — Kiểm tra tính ổn định của trọng số
# =====================================================================
def score_with_weights(df, w_chip, w_ram, w_storage,
                        w_chip_apple, w_ram_apple, w_storage_apple):
    def mms(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)
    norm_chip = mms(df['chip_rating'])
    norm_ram  = mms(df['ram_gb'])
    norm_stor = mms(df['storage_gb'])
    is_apple  = df['brand'] == 'Apple'
    perf = np.where(is_apple,
        (norm_chip*w_chip_apple + norm_ram*w_ram_apple + norm_stor*w_storage_apple)*10,
        (norm_chip*w_chip       + norm_ram*w_ram       + norm_stor*w_storage      )*10)
    return df.assign(score_perf_test=perf.round(2)) \
             .nlargest(5, 'score_perf_test')[['product_name','brand','score_perf_test']]

df_test = df_final.copy()

scenarios = {
    'Baseline (0.50/0.30/0.20 | Apple: 0.65/0.15/0.20)': (0.50, 0.30, 0.20, 0.65, 0.15, 0.20),
    'Tang chip weight (+20%): (0.60/0.25/0.15 | Apple: 0.75/0.10/0.15)': (0.60, 0.25, 0.15, 0.75, 0.10, 0.15),
    'Tang RAM weight  (+20%): (0.40/0.45/0.15 | Apple: 0.55/0.30/0.15)': (0.40, 0.45, 0.15, 0.55, 0.30, 0.15),
}

top5_sets = []
for label, weights in scenarios.items():
    top5 = score_with_weights(df_test, *weights)
    names = set(top5['product_name'].tolist())
    top5_sets.append(names)
    print(f'\n--- {label} ---')
    print(top5.to_string(index=False))

overlap_01 = len(top5_sets[0] & top5_sets[1])
overlap_02 = len(top5_sets[0] & top5_sets[2])
print(f'\n=== Overlap top-5 ===')
print(f'Baseline vs Tang chip:  {overlap_01}/5 phones giong nhau')
print(f'Baseline vs Tang RAM:   {overlap_02}/5 phones giong nhau')
print('\n→ Ket qua on dinh → Weights duoc chon la hop ly.')



--- Baseline (0.50/0.30/0.20 | Apple: 0.65/0.15/0.20) ---
                      product_name   brand  score_perf_test
 Samsung Galaxy S26 Ultra 16GB 1TB Samsung             8.99
       Xiaomi 17 Ultra 5G 16GB 1TB  Xiaomi             8.99
       Xiaomi 15 Ultra 5G 16GB 1TB  Xiaomi             8.68
iPhone 17 Pro Max 2TB | Chính hãng   Apple             8.66
           OPPO Find N6 16GB 512GB    OPPO             8.49

--- Tang chip weight (+20%): (0.60/0.25/0.15 | Apple: 0.75/0.10/0.15) ---
                      product_name   brand  score_perf_test
 Samsung Galaxy S26 Ultra 16GB 1TB Samsung             9.24
       Xiaomi 17 Ultra 5G 16GB 1TB  Xiaomi             9.24
iPhone 17 Pro Max 2TB | Chính hãng   Apple             8.98
           OPPO Find N6 16GB 512GB    OPPO             8.87
     Xiaomi 17 Ultra 5G 16GB 512GB  Xiaomi             8.87

--- Tang RAM weight  (+20%): (0.40/0.45/0.15 | Apple: 0.55/0.30/0.15) ---
                     product_name   brand  score_perf_test
Samsung Gala

### Kết quả Sensitivity Analysis

| Kịch bản | Overlap top-5 |
|----------|---------------|
| Baseline vs Tăng chip +20% | **5/5** |
| Baseline vs Tăng RAM +20% | **5/5** |

**Kết luận:** Top-5 không thay đổi khi weights biến động ±20%.

## Xuất file

In [80]:
output_path = "../../data/final/scored_data.csv"

df_ready_for_kbrs = df_ready_for_kbrs[
    df_ready_for_kbrs['product_name'].notna() &
    (df_ready_for_kbrs['product_name'].str.strip().str.len() > 2) &
    df_ready_for_kbrs['price_vnd'].notna() &
    df_ready_for_kbrs['brand'].notna()
].reset_index(drop=True)

df_ready_for_kbrs.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved {len(df_ready_for_kbrs)} rows to {output_path}")

Saved 387 rows to ../../data/final/scored_data.csv
